In [1]:
from typing import Dict
from data import load_data, time_train_test_split
from config import path_str, TARGET_COL, FEATURE_LEVEL_CONFIGS
from evaluation import eval_best_config_on_holdout
from training import (
    SortFeaturesByCorrElastic,
    SortFeaturesByImportanceXGB,
    build_stability_ranking_elastic,
)

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("error", category=FutureWarning)

# -----------------------------
# Load & basic split
# -----------------------------
df_raw = load_data(path_str.TRAIN_DIR)
df_cut_raw = df_raw[1006:]

X = df_cut_raw.copy()
y = df_cut_raw[TARGET_COL]

X_train, X_test, y_train, y_test = time_train_test_split(X, y)

MODEL_TYPES = ["ols", "ridge", "lasso", "elastic", "lgbm", "xgb"]

# -----------------------------
# 1) Corr / importance rankings (old behaviour)
# -----------------------------
FEATURE_RANKINGS_CORR: Dict[str, list] = {}

for mt in MODEL_TYPES:
    if mt in ["ols", "ridge", "lasso", "elastic"]:
        FEATURE_RANKINGS_CORR[mt] = SortFeaturesByCorrElastic(X_train, y_train)
    elif mt in ["xgb", "lgbm"]:
        FEATURE_RANKINGS_CORR[mt] = SortFeaturesByImportanceXGB(X_train, y_train)
    else:
        FEATURE_RANKINGS_CORR[mt] = []

# -----------------------------
# 2) Stability ranking (ElasticNet-based, shared across models)
# -----------------------------
# This uses the same logic as your training pipeline with prune_mode="stable"
stability_ranking, stability_metrics = build_stability_ranking_elastic(
    X=X_train,
    y=y_train,
    n_splits=3,          # keep in sync with what you use in tuning
    feature_level="extensive",
    verbose=True,
)

# reuse the same stability-based ranking for all models
FEATURE_RANKINGS_STABLE: Dict[str, list] = {
    mt: stability_ranking for mt in MODEL_TYPES
}

# -----------------------------
# Evaluation config
# -----------------------------
# You can add "prune_stable" here once you've run Optuna with that label.
LABELS = ["final", "prune", "broad_tests"]  # e.g. add "prune_stable" if you use it
RESULTS_DIR = "optuna_results"

all_results = []

for label in LABELS:
    for feat_level in FEATURE_LEVEL_CONFIGS:
        for model_type in MODEL_TYPES:
            # Decide which feature ranking to use for this label
            # - "prune"  -> corr/importance-based pruning
            # - "prune_stable"/"stable" -> stability-based pruning
            # - everything else -> no pruning (or corr ranking if selector__k present)
            if label in {"prune_stable", "stable"}:
                feature_ranking = FEATURE_RANKINGS_STABLE[model_type]
            elif label == "prune":
                feature_ranking = FEATURE_RANKINGS_CORR[model_type]
            else:
                # For labels like "final" / "broad_tests" we generally didn't prune.
                # eval_best_config_on_holdout will only touch this if selector__k
                # exists in the best params, otherwise it's effectively ignored.
                feature_ranking = FEATURE_RANKINGS_CORR.get(model_type, [])

            print(f"Evaluating {model_type} ({label}, {feat_level}) on holdout...")

            try:
                res = eval_best_config_on_holdout(
                    model_type=model_type,
                    label=label,
                    X=X,
                    y=y,
                    feature_rankings=feature_ranking,
                    results_dir=RESULTS_DIR,
                    feat_level=feat_level,
                    test_frac=0.2,
                )
                all_results.append(res)
            except FileNotFoundError:
                print(f"No trials file for {model_type} ({label}, {feat_level}), skipping.")
            except Exception as e:
                print(f"[ERROR] {model_type} ({label}, {feat_level}): {e}")

# -----------------------------
# Aggregate & pivot results
# -----------------------------
results_df = pd.DataFrame(all_results)
if not results_df.empty:
    results_df.sort_values(["r2_holdout"], ascending=False, inplace=True)
    print(results_df)

    pivot_r2 = results_df.pivot_table(
        index="model_type",
        columns=["label", "feature_level"],
        values="r2_holdout",
        aggfunc="max",   # or "mean" if you ever duplicate exact combos
    )

    pivot_r2.to_csv("pivot_r2.csv")
    print(pivot_r2)
else:
    print("No results collected; check your LABELS / files on disk.")


c:\Users\lhkke\Documents\HullTactical\HullTactical\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[StabilityRanking] Built ranking for 4650 features over 3 folds.
[StabilityRanking] Top 10 features by stability_score:
                 stability_score  freq_nonzero  sign_consistency
M4_x_vol_high           0.000309      1.000000               1.0
MOM_y_ema_fast          0.000098      1.000000               1.0
V13                     0.000093      1.000000               1.0
M4_lag_1                0.000067      0.666667               1.0
D8_lag_1                0.000046      0.666667               1.0
P6_lag_21               0.000035      0.333333               1.0
D7_x_trend_bear         0.000028      0.333333               1.0
P10_roc_5               0.000026      0.333333               1.0
P5_roc_21               0.000026      0.333333               1.0
MOM_y_roc_63            0.000024      0.333333               1.0
Evaluating ols (final, none) on holdout...
Loading trials from: optuna_results\ols_none_final_trials.csv
No trials file for ols (final, none), skipping.
Evaluating r